# Explore data quality metrics from the pipeline event log

Each pipeline can be configured to save out the metrics to a table in Unity Catalog. From this table we can see what is happening and the quality of the data passing through it.
You can leverage the expecations directly as a SQL table with Databricks SQL to track your expectation metrics and send alerts as required. 
This notebook extracts and analyses expectation metrics to build such KPIS.

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-engineering&org_id=2162748966026566&notebook=%2F1-sdp-sql%2Fexplorations%2F02-Pipeline-event-monitoring&demo_name=pipeline-bike&event=VIEW&path=%2F_dbdemos%2Fdata-engineering%2Fpipeline-bike%2F1-sdp-sql%2Fexplorations%2F02-Pipeline-event-monitoring&version=1">


## Your event log table is now available as a Table within your schema!
This is simply set as an option in your pipeline configuration menu.

In [0]:
%sql
SELECT
  *
FROM
  main.dbdemos_pipeline_bike.pipeline_bike_event_logs
limit 10

id sequence origin timestamp message level maturity_level error details event_type f43546b0-739c-11f0-a591-8e4c99bc3730 List(List(execution, 1754577835925001), null) List(AWS, us-west-2, 1660015457675682, null, c4ebeb89-722c-4bd1-a08e-d7a5022487c8, WORKSPACE, dbdemos_build_pipeline_bike, 0807-144329-mckg7963-v2n, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, null, null, null, null, null, null) 2025-08-07T14:43:59.771Z Update 783e8e is INITIALIZING. INFO STABLE null {"update_progress":{"state":"INITIALIZING"}} update_progress d8145930-739c-11f0-9032-5acaf5d6d5f2 List(List(execution, 1754577835925003), 1754577792581001) List(AWS, us-west-2, 1660015457675682, null, c4ebeb89-722c-4bd1-a08e-d7a5022487c8, WORKSPACE, dbdemos_build_pipeline_bike, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null) 2025-08-07T14:43:12.579Z User quentin.ambard@databricks.com created pipeline. INFO STABLE null {"user_action":{"action":"CREATE","user_name":"quentin.ambard@databricks.com","user_id":7644138420879474,"request":{"create_request":{"id":"c4ebeb89-722c-4bd1-a08e-d7a5022487c8","pipeline_type":"WORKSPACE","name":"dbdemos_build_pipeline_bike","configuration":{"catalog":"main__build","schema":"dbdemos_pipeline_bike"},"libraries":[{"file":{"path":"/Repos/quentin.ambard@databricks.com/dbdemos-notebooks/product_demos/Delta-Live-Table/declarative-pipelines/transformations/01-bronze.sql"}},{"file":{"path":"/Repos/quentin.ambard@databricks.com/dbdemos-notebooks/product_demos/Delta-Live-Table/declarative-pipelines/transformations/02-silver.sql"}},{"file":{"path":"/Repos/quentin.ambard@databricks.com/dbdemos-notebooks/product_demos/Delta-Live-Table/declarative-pipelines/transformations/03-gold.sql"}}],"schema":"dbdemos_pipeline_bike","continuous":false,"development":true,"photon":true,"channel":"CURRENT","catalog":"main__build","serverless":true,"effective_budget_policy_id":"4635ae18-e8d8-4528-98d3-05805c7e6308","event_log":{"name":"pipeline_bike_event_logs","schema":"dbdemos_pipeline_bike","catalog":"main__build"}}}}} user_action e1f90090-739c-11f0-9032-5acaf5d6d5f2 List(List(execution, 1754577835925004), 1754577809178001) List(AWS, us-west-2, 1660015457675682, null, c4ebeb89-722c-4bd1-a08e-d7a5022487c8, WORKSPACE, dbdemos_build_pipeline_bike, null, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, null, null, null, null, null, null) 2025-08-07T14:43:29.177Z User quentin.ambard@databricks.com started an update. INFO STABLE null {"user_action":{"action":"START","user_name":"quentin.ambard@databricks.com","user_id":7644138420879474,"request":{"start_request":{"full_refresh":true,"validate_only":false,"explore_only":false}}}} user_action e201b320-739c-11f0-9032-5acaf5d6d5f2 List(List(execution, 1754577835925006), 1754577809236001) List(AWS, us-west-2, 1660015457675682, null, c4ebeb89-722c-4bd1-a08e-d7a5022487c8, WORKSPACE, dbdemos_build_pipeline_bike, null, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, 783e8eb4-f9ac-4f55-9396-8203b860ce92, null, null, null, null, null, null, null, null, null, null, null, null) 2025-08-07T14:43:29.234Z Update 783e8e started by API_CALL. INFO STABLE null {"create_update":{"cause":"API_CALL","config":{"id":"c4ebeb89-722c-4bd1-a08e-d7a5022487c8","pipeline_type":"WORKSPACE","name":"dbdemos_build_pipeline_bike","configuration":{"catalog":"main__build","pipelines.acfsProcessorVersion":"1","pipelines.allowClearingTableComment":"true","pipelines.allowPrivateToNonPrivateAltersWithCustomSchemaPreview":"true","pipelines.alterExistingMvSt":"true","pipelines.alterViewsInHMS.enabled":"true","pipelines.alterableMetadataBehavior":"preserve","pipelines.analysis.allowSqlFlowWithConfParallelResolution":"true","pipelines.analysis.maybeResolveFlowsPar

The `details` column contains metadata about each Event sent to the Event Log in a JSON blob. Using `parse_json` and the `VARIANT` data type we can explore it as if it was an object. There are different fields depending on what type of Event it is. Some examples include:
* `user_action` Events occur when taking actions like creating the pipeline
* `flow_definition` Events occur when a pipeline is deployed or updated and have lineage, schema, and execution plan information
  * `output_dataset` and `input_datasets` - output table/view and its upstream table(s)/view(s)
  * `flow_type` - whether this is a complete or append flow
  * `explain_text` - the Spark explain plan
* `flow_progress` Events occur when a data flow starts running or finishes processing a batch of data
  * `metrics` - currently contains `num_output_rows`
  * `data_quality` - contains an array of the results of the data quality rules for this particular dataset
    * `dropped_records`
    * `expectations`
      * `name`, `dataset`, `passed_records`, `failed_records`
  

In [0]:
%sql
SELECT
  details:flow_definition.output_dataset,
  details:flow_definition.input_datasets,
  details:flow_definition.flow_type,
  details:flow_definition.schema,
  details:flow_definition
FROM main.dbdemos_pipeline_bike.pipeline_bike_event_logs
WHERE details:flow_definition IS NOT NULL
ORDER BY timestamp


output_dataset,input_datasets,flow_type,schema,flow_definition
main.dbdemos_pipeline_bike.maintenance_logs_raw,null,APPEND,"[{""name"":""maintenance_id"",""path"":[""maintenance_id""],""data_type"":""STRING""},{""name"":""bike_id"",""path"":[""bike_id""],""data_type"":""STRING""},{""name"":""reported_time"",""path"":[""reported_time""],""data_type"":""TIMESTAMP""},{""name"":""resolved_time"",""path"":[""resolved_time""],""data_type"":""DATE""},{""name"":""issue_description"",""path"":[""issue_description""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""}]","{""output_dataset"":""main.dbdemos_pipeline_bike.maintenance_logs_raw"",""explain_text"":""'Project [*]"",""schema_json"":""{\""type\"":\""struct\"",\""fields\"":[{\""name\"":\""maintenance_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""bike_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""reported_time\"",\""type\"":\""timestamp\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""resolved_time\"",\""type\"":\""date\"",\""nullable\"":true,\""metadata\"":{\""__detected_date_formats\"":\""yyyy-M-d\""}},{\""name\"":\""issue_description\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""_rescued_data\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}}]}"",""schema"":[{""name"":""maintenance_id"",""path"":[""maintenance_id""],""data_type"":""STRING""},{""name"":""bike_id"",""path"":[""bike_id""],""data_type"":""STRING""},{""name"":""reported_time"",""path"":[""reported_time""],""data_type"":""TIMESTAMP""},{""name"":""resolved_time"",""path"":[""resolved_time""],""data_type"":""DATE""},{""name"":""issue_description"",""path"":[""issue_description""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""}],""flow_type"":""APPEND"",""comment"":""Raw maintenance logs streamed in from CSV files."",""language"":""SQL"",""notebook_path"":""/Repos/quentin.ambard@databricks.com/dbdemos-notebooks/product_demos/Delta-Live-Table/declarative-pipelines/transformations/01-bronze.sql"",""once"":false}"
main.dbdemos_pipeline_bike.customers_cdc_raw,null,APPEND,"[{""name"":""customer_id"",""path"":[""customer_id""],""data_type"":""STRING""},{""name"":""user_type"",""path"":[""user_type""],""data_type"":""STRING""},{""name"":""registration_date"",""path"":[""registration_date""],""data_type"":""STRING""},{""name"":""email"",""path"":[""email""],""data_type"":""STRING""},{""name"":""phone"",""path"":[""phone""],""data_type"":""STRING""},{""name"":""age_group"",""path"":[""age_group""],""data_type"":""STRING""},{""name"":""membership_tier"",""path"":[""membership_tier""],""data_type"":""STRING""},{""name"":""preferred_payment"",""path"":[""preferred_payment""],""data_type"":""STRING""},{""name"":""home_station_id"",""path"":[""home_station_id""],""data_type"":""STRING""},{""name"":""is_active"",""path"":[""is_active""],""data_type"":""BOOLEAN""},{""name"":""operation"",""path"":[""operation""],""data_type"":""STRING""},{""name"":""event_timestamp"",""path"":[""event_timestamp""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""}]","{""output_dataset"":""main.dbdemos_pipeline_bike.customers_cdc_raw"",""explain_text"":""'Project [*]"",""schema_json"":""{\""type\"":\""struct\"",\""fields\"":[{\""name\"":\""customer_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""user_type\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""registration_date\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""email\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""phone\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""age_group\"",\""type\"":\""string\"",\""nullable\"":true,\""me

In [0]:
%sql
select
  e.origin.update_id,
  ex.value:name::string,
  ex.value:dataset::string,
  ex.value:passed_records::long as passed_records,
  ex.value:failed_records::long as failed_records
from
  main.dbdemos_pipeline_bike.pipeline_bike_event_logs e,
  lateral variant_explode(parse_json(e.details:flow_progress:data_quality:expectations:[ * ])) as ex
where
  e.event_type = "flow_progress"
  and details:flow_progress:status = "RUNNING"
  and details:flow_progress:data_quality:expectations IS NOT NULL

update_id,name,dataset,passed_records,failed_records
783e8eb4-f9ac-4f55-9396-8203b860ce92,invalid_ride_duration,main.dbdemos_pipeline_bike.rides,35485,376
783e8eb4-f9ac-4f55-9396-8203b860ce92,short_maintenance_description,main.dbdemos_pipeline_bike.maintenance_logs,611,62
783e8eb4-f9ac-4f55-9396-8203b860ce92,no_maintenance_description,main.dbdemos_pipeline_bike.maintenance_logs,649,24


## Tracking data quality as an AI/BI dashboard

Let's leverage Databricks AI/BI dashboard to monitor our pipeline and data ingestion. 

- Open the <a  dbdemos-dashboard-id="data-quality" href='/sql/dashboardsv3/01f1b2d4d9d7130aa27fc3d701686c8e' target="_blank">Bike Rental Data Monitoring Dashboard</a> to track all your data quality, and add alerts based on your requirements.
- Open the <a  dbdemos-dashboard-id="operational" href='/sql/dashboardsv3/01f1b2d4d9a319c194b0b0e7fc86733f' target="_blank">Bike Rental Operational Pipeline Dashboard</a> to track all your pipeline event and cost!